<a id="top"></a>

# Lab L0707: trace a tool-call round-trip

```yaml
title: "Lab L0707: trace a tool-call round-trip"
keywords: round-trip, four messages, tool_calls, ToolMessage, call id, three outcomes, trace, lab
estimated duration: 30
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0707_lab_solutions.ipynb`. **Pure Python — no API key needed** (you dissect crafted transcripts, the same offline approach as the L0706 demo).
>
> **After this lab you can:** read a tool-call transcript message by message; match a `ToolMessage` to the tool call it answers by id; and tell the three observable outcomes apart.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Summarize what each message carries](#2-problem-1--summarize-what-each-message-carries)
- [3. Problem 2 — Match the result to the request by id](#3-problem-2--match-the-result-to-the-request-by-id)
- [4. Problem 3 — Tell the three outcomes apart](#4-problem-3--tell-the-three-outcomes-apart)
- [5. Problem 4 — Why four messages? (written)](#5-problem-4--why-four-messages-written)
- [6. Self-check](#6-self-check)

## 1. Setup

A captured four-message transcript of one successful round-trip, written as LangChain **message objects** (`HumanMessage`, `AIMessage` with `tool_calls`, `ToolMessage`).

In [ ]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage

CALL_ID = "toolu_01XYZ"
transcript: list[BaseMessage] = [
    HumanMessage("What is 6,150 multiplied by 7,012?"),
    AIMessage(
        content="",
        tool_calls=[
            {
                "id": CALL_ID,
                "name": "calculator",
                "args": {"expression": "6150 * 7012"},
                "type": "tool_call",
            }
        ],
    ),
    ToolMessage(content="43123800", tool_call_id=CALL_ID),
    AIMessage(content="6,150 x 7,012 = 43,123,800."),
]
print(f"{len(transcript)} messages captured")

[↑ Back to top](#top)

## 2. Problem 1 — Summarize what each message carries

Implement `describe(msg)` returning what a message carries: its tool calls (an `AIMessage` with a non-empty `.tool_calls`), a tool result (a `ToolMessage`), or plain text. Print one line per message: its `.type` and that summary.

In [ ]:
def describe(msg: BaseMessage) -> str:
    """Summarize what one message carries: its tool calls, a tool result, or text."""
    if isinstance(msg, AIMessage) and msg.tool_calls:
        return "tool_calls: " + ", ".join(c["name"] for c in msg.tool_calls)
    if isinstance(msg, ToolMessage):
        return "tool result"
    return "text"


for i, msg in enumerate(transcript, start=1):
    print(f"msg {i}: type={msg.type:5} {describe(msg)}")

[↑ Back to top](#top)

## 3. Problem 2 — Match the result to the request by id

Implement `ids_match(transcript)`: pull the tool call's `id` from message 2's `AIMessage.tool_calls`, and the `tool_call_id` from message 3's `ToolMessage`, and return whether they are equal. That id is what ties a result to the call it answers.

In [ ]:
def ids_match(t: list[BaseMessage]) -> bool:
    """True if the ToolMessage in msg 3 answers the tool call in msg 2's AIMessage."""
    ai = t[1]
    tool = t[2]
    assert isinstance(ai, AIMessage) and isinstance(tool, ToolMessage)
    return ai.tool_calls[0]["id"] == tool.tool_call_id


print("ids match:", ids_match(transcript))  # expect True

[↑ Back to top](#top)

## 4. Problem 3 — Tell the three outcomes apart

Handing a model a tool has three observable outcomes. Implement `classify(msg)` for an `AIMessage` returning `"called"` (has a tool call), `"answered"` (text only, no tool call), or `"malformed"` (a tool call whose `args` is missing the `expression` key). Run it over the three crafted `AIMessage`s.

In [ ]:
CASES: dict[str, AIMessage] = {
    "A": AIMessage(
        content="",
        tool_calls=[
            {"id": "t1", "name": "calculator", "args": {"expression": "2*3"}, "type": "tool_call"}
        ],
    ),
    "B": AIMessage(content="It is 6."),
    "C": AIMessage(
        content="",
        tool_calls=[{"id": "t2", "name": "calculator", "args": {}, "type": "tool_call"}],
    ),
}


def classify(msg: AIMessage) -> str:
    """called (has a tool call) / answered (text, no call) / malformed (call missing 'expression')."""
    if not msg.tool_calls:
        return "answered"
    if "expression" not in msg.tool_calls[0]["args"]:
        return "malformed"
    return "called"


for name, msg in CASES.items():
    print(name, "->", classify(msg))

[↑ Back to top](#top)

## 5. Problem 4 — Why four messages? (written)

The user asked **once**, but a successful tool exchange is at least four messages. In a sentence or two: what are the four messages, and who produces each?

_Write your answer by editing this cell (double-click)._

The four messages are: (1) a `HumanMessage` — the user's question; (2) an `AIMessage` carrying a tool call — the model's request; (3) a `ToolMessage` — the tool's output; and (4) a final `AIMessage` — the model's natural-language answer. The human speaks once (message 1), the **model** produces messages 2 and 4, and the **application** produces message 3 when it runs the function. The `ToolMessage` is your code speaking, not the human.

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0707_lab_solutions.ipynb`. Done when `ids_match` is `True` and `classify` returns `called`, `answered`, `malformed` for A, B, C.

[↑ Back to top](#top)